<a href="https://colab.research.google.com/github/Susan-sapkota-98/Nepali-asr-evaluation/blob/main/04_cleaning_rules.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q jiwer

In [2]:
from pathlib import Path

import jiwer
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

samples_folder = Path("/content/drive/MyDrive/nepali-asr-evaluation/samples_20")
samples = pd.read_csv(samples_folder / "labeled_20.csv", dtype=str, keep_default_na=False)
print(len(samples), "rows")

Mounted at /content/drive
20 rows


In [3]:
digit_map = str.maketrans("०१२३४५६७८९","0123456789")

def clean_text(text):
  text=text.translate(digit_map)
  text= " ".join(text.split())
  return text

In [6]:
print(clean_text("२०३२ साल"))

2032 साल


In [8]:
def score_rows(correct_list, predicted_list):
    wers = []
    cers = []
    for correct, predicted in zip(correct_list, predicted_list):
        wers.append(jiwer.wer(correct, predicted))
        cers.append(jiwer.cer(correct, predicted))
    return wers, cers


# before cleaning
samples["raw_wer"], samples["raw_cer"] = score_rows(samples["text"], samples["mms_prediction"])

# after cleaning
clean_correct = [clean_text(t) for t in samples["text"]]
clean_predicted = [clean_text(t) for t in samples["mms_prediction"]]
samples["clean_wer"], samples["clean_cer"] = score_rows(clean_correct, clean_predicted)

# CER with all spaces removed
no_space_correct = [t.replace(" ", "") for t in clean_correct]
no_space_predicted = [t.replace(" ", "") for t in clean_predicted]
no_space_wer, no_space_cer = score_rows(no_space_correct, no_space_predicted)
samples["no_space_cer"] = no_space_cer

In [9]:
print("raw WER:", round(jiwer.wer(list(samples["text"]), list(samples["mms_prediction"])) * 100, 1), "%")
print("clean WER:", round(jiwer.wer(clean_correct, clean_predicted) * 100, 1), "%")
print("raw CER:", round(jiwer.cer(list(samples["text"]), list(samples["mms_prediction"])) * 100, 1), "%")
print("clean CER:", round(jiwer.cer(clean_correct, clean_predicted) * 100, 1), "%")
print("CER without spaces:", round(jiwer.cer(no_space_correct, no_space_predicted) * 100, 1), "%")

raw WER: 41.4 %
clean WER: 41.4 %
raw CER: 20.3 %
clean CER: 19.8 %
CER without spaces: 20.2 %


In [10]:
samples.groupby("my_label")[["raw_wer", "clean_wer", "raw_cer", "clean_cer", "no_space_cer"]].mean().round(2)

,raw_wer,clean_wer,raw_cer,clean_cer,no_space_cer
my_label,,,,,
exact,0.00,0.00,0.00,0.00,0.00
failed,1.00,1.00,2.21,2.21,2.38
number,1.00,1.00,0.31,0.19,0.14
real_mistake,0.57,0.57,0.20,0.20,0.20
spacing,0.67,0.67,0.14,0.14,0.08
spelling,0.48,0.48,0.12,0.12,0.12


In [11]:
changed = samples[samples["raw_wer"] != samples["clean_wer"]]
changed[["text", "mms_prediction", "raw_wer", "clean_wer"]]

,text,mms_prediction,raw_wer,clean_wer


In [12]:
samples.to_csv(samples_folder / "cleaned_20.csv", index=False)
print("saved")

saved
